# PrimeKG covered-entity GoG pilot

This notebook compares a flat ROOT R-GCN, a flat R-GCN with GoG context, and the hierarchical GoG encoder on the **original fixed PrimeKG splits**. It does not re-split GoG relations. The current GoG covers 624 entities and should yield 2,936 training, 226 validation, and 256 test facts. Select a T4 GPU runtime. The repository changes implementing `covered-pilot` must be pushed to GitHub before this clone-based notebook runs.


In [ ]:
# Get the current training code.
from pathlib import Path
import os, subprocess, sys
REPO_URL = "https://github.com/richie955/graphofgraphs.git"
REPO_DIR = Path("/content/graphofgraphs")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
assert (REPO_DIR / "training/covered_pilot.py").is_file(), "Push the covered-pilot code to GitHub, then rerun this cell."
print("Repository:", REPO_DIR)


In [ ]:
# Set the folder containing the four JSON files. Change only DATA_DIR if needed.
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = Path("/content/drive/MyDrive/pilot")
GOG = DATA_DIR / "prime_kg_gog_from_training.json"
TRAIN = DATA_DIR / "train_data.json"
VALIDATION = DATA_DIR / "validation_data.json"
TEST = DATA_DIR / "test_data.json"
for path in (GOG, TRAIN, VALIDATION, TEST):
    assert path.is_file(), f"Missing input file: {path}"
    print(path.name, round(path.stat().st_size / 1024**2, 2), "MiB")


In [ ]:
# Install model dependencies and audit the fixed splits before training.
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch"], check=True)
    import torch
try:
    import torch_geometric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch_geometric"], check=True)
from training.covered_pilot import load_covered_experiment
experiment = load_covered_experiment(GOG, TRAIN, VALIDATION, TEST)
print("GPU available:", torch.cuda.is_available())
print("Audit:", experiment.audit)


In [ ]:
# Full five-seed comparison. Validation chooses each model's checkpoint; test is evaluated afterward.
OUTPUT_DIR = REPO_DIR / "runs/primekg-covered-pilot"
command = [sys.executable, "-m", "training.train", str(GOG),
           "--model", "covered-pilot",
           "--train-facts", str(TRAIN),
           "--validation-facts", str(VALIDATION),
           "--test-facts", str(TEST),
           "--seeds", "17", "23", "42", "73", "101",
           "--device", "auto", "--output-dir", str(OUTPUT_DIR),
           "--epochs", "100", "--patience", "15"]
subprocess.run(command, cwd=REPO_DIR, check=True)
print("Report:", OUTPUT_DIR / "covered_pilot_report.json")


In [ ]:
# Download reports and checkpoints before the Colab runtime ends.
import shutil
from google.colab import files
archive = shutil.make_archive("primekg-covered-pilot-results", "zip", OUTPUT_DIR)
files.download(archive)
